# [Super AI Engineer Season 6] Hackathon Week 4
## 5 Domains Hackathon: Sleep Staging Classification

**Super AI Engineer Season 6 - Level 1 Hackathon**  
- Dataset: Sleep Staging Classification
- จัดทำโดย: 600425-วิศิษฐ์

---
### Notebook Outline
1. Setup & Imports  
2. Data Loading & Initial Inspection  
3. Custom Transforms & Feature Extraction  
4. Model Preparation & Training  
5. Evaluation & Analysis  
6. Prediction & Submission Generation


# 1. Setup & Imports
### 1.1 ติดตั้งและนำเข้าไลบรารีที่จำเป็น

In [1]:
!pip -q install autogluon

import os
import glob
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from autogluon.tabular import TabularDataset, TabularPredictor

KAGGLE_MODE = True

data_dir = "/kaggle/input/competitions/super-ai-engineer-ss-6-sleep-stage-classification"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 10.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.1/452.1 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 5.4 MB/s eta 0:00

# 2. Custom Transforms & Feature Extraction
สร้างฟังก์ชันดึงฟีเจอร์พื้นฐาน อ้างอิงจาก Pipeline ฉบับใหม่ (std_x, std_y, std_z, std_bvp, mean_temp)

In [2]:
def extract_basic_features(df):
    """
    สกัดฟีเจอร์เบื้องต้นที่ให้ผลลัพธ์ครอบคลุมและมีประสิทธิภาพในการทำ Model 
    ได้แก่ Standard Deviation ของ ACC_X, ACC_Y, ACC_Z, BVP และ Mean ของ TEMP
    """
    std_x = df['ACC_X'].std()
    std_y = df['ACC_Y'].std()
    std_z = df['ACC_Z'].std()
    std_bvp = df['BVP'].std()
    mean_temp = df['TEMP'].mean()
    
    return [std_x, std_y, std_z, std_bvp, mean_temp]

# 3. Data Loading & Feature Extraction Pipeline
โหลดข้อมูล Train ทั้งหมด จัดการตัดเป็น Window ละ 480 rows และกำจัด File ที่ Error (เช่นลำดับที่ 38) ทิ้งตาม Pipeline ต้นฉบับ

In [3]:
paths_train = sorted(glob.glob(os.path.join(data_dir, 'train/train/*')))
prepare_data = []
count = 0

for path in tqdm(paths_train, desc="Processing Train Data"):
    # ทำการ Skip file ลำดับที่ 38 เหมือนใน Reference เพื่อเลี่ยงปัญหา Data Quality/Error
    if count + 1 in [38]:
        print('skip:', path)
        count += 1
        continue
        
    df_train = pd.read_csv(path)
    
    # ตัดข้อมูลแบ่งเป็นชิ้นละ 480 วินาที
    for i in range(1, int(len(df_train)/480)):
        df_1920 = df_train.iloc[480*(i-1) : 480*i]
        
        # พยายามดึงค่า Label อย่างปลอดภัย
        raw_label = df_1920['Sleep_Stage'].iloc[0]
        if isinstance(raw_label, str) and len(raw_label) > 1 and raw_label[0] == '[':
            # Handling string representations like '[1]'
            label = int(raw_label.strip('[]'))
        elif isinstance(raw_label, (list, np.ndarray)):
            label = raw_label[0]
        else:
            label = raw_label
            
        features = extract_basic_features(df_1920)
        prepare_data.append(features + [label])
        
    count += 1

print("Finish Feature Extraction for Train!")

Processing Train Data:   0%|          | 0/83 [00:00<?, ?it/s]

skip: /kaggle/input/competitions/super-ai-engineer-ss-6-sleep-stage-classification/train/train/train038.csv
Finish Feature Extraction for Train!


In [4]:
df_train_ok = pd.DataFrame(prepare_data, columns=['std_x', 'std_y', 'std_z', 'std_bvp', 'mean_temp', 'label'])
df_train_ok.head()

,std_x,std_y,std_z,std_bvp,mean_temp,label
0,4.050812,1.309091,4.181039,156.050937,31.728616,W
1,1.612885,0.500470,2.476805,123.266837,31.717368,W
2,0.680005,0.192843,0.984128,52.941762,31.753257,W
3,0.824772,0.262576,1.537755,76.620256,31.840240,W
4,2.989937,0.809021,2.972157,153.803718,31.937764,W


# 4. Model Preparation & Training
อ้างอิงจาก Reference จะเน้นไปที่โมเดลกลุ่ม CatBoost (CAT) ที่ค่อนข้างทำผลงานได้ดีเยี่ยมกับข้อมูลชุดนี้

In [5]:
save_path = 'best_model'
hyperparameters = {
    'CAT': [
        {
            'ag_args_fit': {'num_gpus': 0},
            'iterations': 500,
            'learning_rate': 0.05,
            'depth': 8,
            'l2_leaf_reg': 3,
            'random_strength': 1,
            'bagging_temperature': 1,
            'border_count': 128
        },
        {
            'ag_args_fit': {'num_gpus': 1},
            'iterations': 1000,
            'learning_rate': 0.03,
            'depth': 10,
            'l2_leaf_reg': 5,
            'random_strength': 1.5,
            'bagging_temperature': 1.5,
            'border_count': 256,
            'task_type': 'GPU'
        }
    ],
}
time_limit = 120

In [6]:
predictor = TabularPredictor(label='label', 
                             problem_type='multiclass',
                             path=save_path).fit(
    df_train_ok,
    presets='best_quality',
    hyperparameters=hyperparameters,
    time_limit=time_limit
)
print("Model Training Finished!")

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Sat Jan 17 11:20:45 UTC 2026
CPU Count:          4
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB | GPU 1: 14.56/14.56 GB
Total GPU Memory:   Free: 29.13 GB, Allocated: 0.00 GB, Total: 29.13 GB
GPU Count:          2
Memory Avail:       29.55 GB / 31.35 GB (94.3%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable

Model Training Finished!


# 5. Evaluation & Analysis
ตรวจสอบประสิทธิภาพของ Model บนชุดข้อมูล Evaluate

In [7]:
predictor.leaderboard()

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,CatBoost_BAG_L1,0.60111,accuracy,0.502452,68.919627,0.502452,68.919627,1,True,1
1,WeightedEnsemble_L2,0.60111,accuracy,0.508487,68.936658,0.006035,0.017031,2,True,2


# 6. Prediction & Submission Generation
อ่านไฟล์ `sample_submission.csv` เป็นแกนหลัก แล้ว Match ลำดับการอ่าน Test เพื่อไม่ให้เกิดอาการ Predict ข้ามแถว

In [8]:
sample_submission = pd.read_csv(f"{data_dir}/sample_submission.csv")
prepare_data_test = []

with tqdm(total=len(sample_submission), desc="Processing Test Files") as pbar:
    for index, row in sample_submission.iterrows():
        # แยก ID เพื่อหา Path ที่เชื่อมโยงไปที่ csv ไฟล์จริง
        id_parts = row['id'].split("_")
        path = os.path.join(data_dir, f"test_segment/test_segment/{id_parts[0]}/{id_parts[0]}_{id_parts[1]}.csv")
        
        sub_df_test = pd.read_csv(path)
        features = extract_basic_features(sub_df_test)
        prepare_data_test.append(features)
        pbar.update(1)
        
df_test_ok = pd.DataFrame(prepare_data_test, columns=['std_x', 'std_y', 'std_z', 'std_bvp', 'mean_temp'])
print("Finish Extracting Test Data! Shape:", df_test_ok.shape)

Processing Test Files:   0%|          | 0/7832 [00:00<?, ?it/s]

Finish Extracting Test Data! Shape: (7832, 5)


In [9]:
y_pred = predictor.predict(df_test_ok)
sample_submission['labels'] = y_pred

display(sample_submission['labels'].value_counts())
display(sample_submission.head())

labels
N2    6247
W     1549
R       20
N1      15
N3       1
Name: count, dtype: int64

,id,labels
0,test001_00000,W
1,test001_00001,N2
2,test001_00002,N2
3,test001_00003,N2
4,test001_00004,W


In [10]:
sample_submission.to_csv("5hack_ssc_submission.csv", index=False)
print("Submission file exported successfully!")

Submission file exported successfully!
